# Suite XPRS — Parsing C++ into the store

`Cpp.Parse.parse(text, path)` reads C++ source with tree-sitter-cpp and gives the callback of the transaction that
writes it: one upsert, binding the workspace and the translation unit, that files the unit in the workspace by its
path and writes the unit's `body` as one value. The whole tree is values: each declaration and statement is a value
object, a branch of `Programs.Cpp.Statement` tagged by its kind in lower case, with the line it begins on (`line`),
holding the statements of its blocks in its properties, in order; C's kinds are C++'s by name, over C++'s terms.
Expressions are mbse-expressions' `Cpp` terms, a declaration's specifiers, declarators and attributes its helpers,
type subterms `Cpp Types`' and directives `C Preprocessor`'s; where a term holds statements or declarations, its
`block`, `members` or `group` holds them. Each case parses constructs into one store, reads the unit's tree back as
plain data within a transaction (a term or a helper as the printer writes it, or a term as its kind, attributes and
subterms), and checks it; then invalid syntax, which is kept, and the one error, text that is no text; then each place
where tree-sitter-cpp departs from C++, by the number `Parse` gives it. Its cases port 0.3's C++ cases of PRS
(`03_Parse.ipynb`) and CPP (`02_Ccpp.ipynb`).

In [ ]:
import itertools
import json
from pathlib import Path

from mbse.Expressions.Cpp.Expressions import Expressions as CppExpressions
from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import Cpp
from mbse.Programs.Cpp import Parse, Print
from mbse.Programs.Cpp.Statements import KINDS, PARTS
from mbse.Programs.Framework import Model
from mbse.Programs.Framework.Errors import ParseError
from mbse.Schemas.Framework import Plain, References, Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Cpp.store()
PATHS = (f"case{n}.cpp" for n in itertools.count())
HOLDERS = ("block", "members", "group")  # the properties of terms and helpers that hold this grammar's statements


def shape(term):
    """A term as plain data: its kind, its attributes (a helper as its kind and attributes), and its subterms."""
    def part(value):
        if isinstance(value, Terms.Part):
            return (value.kind, {k: part(v) for k, v in value.attributes.items()})
        return [part(v) for v in value] if isinstance(value, tuple) else value
    tag, record = Terms.kind_of(term)
    attributes = {k: part(v) for k, v in Terms.attributes_of(record).items()}
    held = Objects.properties(record)
    attributes.update({name: plain(held[name]) for name in HOLDERS if name in held})
    return (tag, attributes, [None if a is None else shape(a) for a in Terms.arguments_of(record)])


def plain(value, shapes=False):
    """What a property holds, as plain data: a term as the printer writes it (or its `shape`), a helper so too, a
    container as a list, a union's value as its branch's, a value object as a dict of its kind and its properties."""
    if isinstance(value, Objects.Listed):
        return [plain(v, shapes) for v in value.values]
    if Terms.is_term(value):
        return shape(value) if shapes else Print.Printer().term(value)
    if Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        record = Terms.kind_of(value)[1]  # a branch of this grammar's, a term (a condition), or a helper
        if Terms.is_term(record) or record.as_schema().name.startswith("Programs."):
            return plain(record, shapes)
        return Print.Printer().written(value)
    if Objects.is_object(value) and value.as_schema().name.startswith("Expressions."):
        return Print.Printer().written(value)
    if Objects.is_object(value):
        return {"kind": value.as_schema().name.rsplit(".", 1)[-1]} | {
            k: plain(v, shapes) for k, v in Objects.properties(value).items()}
    return value


def unit(store, path):
    """The translation unit at `path`, read within the transaction open on `store`."""
    return store.singleton(Model.WORKSPACE).files({"path": path})


def parse(text, shapes=False):
    """The declarations of the translation unit `text` parses to, written in the store at a path of its own, read
    back."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    with STORE.transaction():
        return plain(Objects.properties(unit(STORE, path))["body"], shapes)


def first(text, shapes=False):
    return parse(text, shapes)[0]


def body(text, shapes=False):
    """The statements of `text`, the body of a function `f`."""
    return first("void f() {\n" + text + "\n}", shapes)["body"]["body"]


def statement(text, shapes=False):
    return body(text, shapes)[0]


def value(text, shapes=False):
    """The expression of the expression statement `text`, within a function."""
    return statement(text + ";", shapes)["expression"]


def members(text):
    """The members of the one class `text` declares, its `members`."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    with STORE.transaction():
        (declaration,) = Objects.properties(unit(STORE, path))["body"].values
        specifier = Terms.kind_of(Objects.properties(Terms.kind_of(declaration)[1])["specifiers"].values[0])[1]
        return plain(Terms.kind_of(Objects.properties(specifier)["members"])[1])["members"]


def kinds(nodes):
    return [n["kind"] for n in nodes]


def printed(text):
    """The text the translation unit `text` parses to prints as."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return Print.print_module(STORE, path)


def same(text):
    """Whether `text`, in the printer's layout, prints back as written."""
    return printed(text) == text


def error(text):
    """Whether `text` is kept as invalid syntax, an `Error` or an `error` term within it, as its frame assigns it."""
    (frame,) = STORE.transaction(Parse.parse(text, next(PATHS)))
    return '"error"' in json.dumps(assigned(STORE, frame))


def assigned(store, frame):
    """What `frame`, given back by a transaction on `store`, assigns, as `References.OfFrame` writes it as plain data."""
    written = Plain.ToPlain(store).Value(store.registered("References.OfFrame"), References.of(store, frame))
    return written["assigns"]

<a id="XPRS-01"></a>
## XPRS-01 · The conformance sources are one transaction each: every statement, declaration, member and clause kind and every part, terms the dialect validates, JSON that reads back and writes the same bytes; C++20's source keeps invalid only what mbse-expressions cannot hold

In [ ]:
def walk(value, outermost=True):
    """The value objects and the outermost terms within `value`, in source order: a term's subterms and helpers are its
    own, but the statements its block, members or group holds are walked as any."""
    if isinstance(value, Objects.Listed):
        for v in value.values:
            yield from walk(v, outermost)
    elif Terms.is_term(value):
        if outermost:
            yield value
        for name, held in Objects.properties(Terms.kind_of(value)[1]).items():
            yield from walk(held, name in HOLDERS)
    elif Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        yield from walk(Terms.kind_of(value)[1], outermost)
    elif Objects.is_object(value):
        if outermost:
            yield value
        for name, held in Objects.properties(value).items():
            yield from walk(held, outermost or name in HOLDERS)


seen, problems, errors = set(), [], []
for name in ("cpp20.cpp", "cpp23.cpp"):
    text = (SOURCES / name).read_text(encoding="utf-8")
    (frame,) = STORE.transaction(Parse.parse(text, name))
    assert sorted(frame) == ["unit", "ws"]  # the workspace and the unit; every declaration is a value within it
    (assign,) = assigned(STORE, frame)  # the unit, given its whole tree as one nested value
    with STORE.transaction():
        for node in walk(Objects.properties(unit(STORE, name))["body"]):
            if Terms.is_term(node):  # a literal's type is read under an ABI, which only an expression's holder names
                problems.extend(p for p in CppExpressions.DIALECT.validate(node) if "names no ABI" not in p and (
                    "is not bound" not in p))
            elif node.as_schema().name == "Programs.Cpp.Error":
                errors.append((name, node.text.get()))
            else:
                seen.add(node.as_schema().name.rsplit(".", 1)[-1])
    # Written as plain JSON, read into a fresh store holding the installs, executed, and written again: the same bytes
    written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse(text, name, STORE))))
    fresh = Cpp.store()
    (again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
    assert sorted(again) == sorted(frame) and assigned(fresh, again) == assigned(STORE, frame)
    assert json.dumps(Plain.ToPlain(fresh).Transaction(fresh.Mutations.OfTransaction(
        Parse.parse(text, name, fresh)))) == written
assert set(KINDS) <= seen, set(KINDS) - seen  # every statement, declaration, member and clause kind
assert set(PARTS) <= seen, set(PARTS) - seen  # and every part
assert problems == [], problems[:5]
assert errors == [  # what mbse-expressions' C++ cannot hold (Parse's 38 and 39): its gaps, reported to it
    ("cpp20.cpp", "explicit(N > 1) Vector(T a, T b);"), ("cpp20.cpp", "auto [first, second] = std::pair{1, 2};"),
    ("cpp20.cpp", "for (auto &[key, value] : map) {}")]
print(len(seen), "kinds")

<a id="XPRS-02"></a>
## XPRS-02 · The transaction: one upsert by the file's path, binding the workspace and the unit, its tree one nested value of C++'s kinds, C's among them by name; a lambda's statements in its `block`, a class's members in its `members`, a conditional's lines in its `group`

In [ ]:
source = "int f(int a) {\n    if (a) {\n    }\n    return [] {\n        return 1;\n    }();\n}\n"
(frame,) = STORE.transaction(Parse.parse(source, "shape.cpp"))
assert sorted(frame) == ["unit", "ws"]
assert set(frame.assigns) == {"unit"} and set(frame.assigns["unit"]) == {"body"}
(assign,) = assigned(STORE, frame)
definition = assign["properties"]["body"]["Programs.Cpp.Body"][0]["functiondefinition"]
assert definition["body"]["compoundstatement"]["body"][0] == {"ifstatement": {
    "line": 2, "condition": {"expression": {"idexpression": {"name": {"identifier": {"spelling": "a"}}}}},
    "consequence": {"compoundstatement": {"line": 2}}}}
with STORE.transaction():
    file = unit(STORE, "shape.cpp")
    assert file.as_schema().name == "Programs.Cpp.TranslationUnit"
    (definition,) = Objects.properties(file)["body"].values
    tag, node = Terms.kind_of(definition)
    assert (tag, node.as_schema().name) == ("functiondefinition", "Programs.Cpp.FunctionDefinition")
    body_ = Terms.kind_of(Objects.properties(node)["body"])[1]
    _, returned = Terms.kind_of(Objects.properties(body_)["body"].values[1])
    call = Terms.kind_of(Objects.properties(returned)["value"])[1]
    lambda_ = Terms.kind_of(Objects.properties(call)["function"])[1]
    # a lambda, a term, holds its statements in its `block`: this grammar's `CompoundStatement`, a branch of the
    # dialect's open union `Expressions.Cpp.Expressions.Block`
    block = Objects.properties(lambda_)["block"]
    assert block.as_schema().name == "Expressions.Cpp.Expressions.Block"
    tag, node = Terms.kind_of(block)
    assert (tag, node.as_schema().name) == ("compoundstatement", "Programs.Cpp.CompoundStatement")
path = next(PATHS)
STORE.transaction(Parse.parse("struct S {\n#ifdef A\n    int a;\n#endif\n};", path))
with STORE.transaction():
    (declaration,) = Objects.properties(unit(STORE, path))["body"].values
    (specifier,) = Objects.properties(Terms.kind_of(declaration)[1])["specifiers"].values
    held = Objects.properties(Terms.kind_of(specifier)[1])["members"]
    assert held.as_schema().name == "Expressions.Cpp.Expressions.Members"
    tag, node = Terms.kind_of(held)
    assert (tag, node.as_schema().name) == ("memberlist", "Programs.Cpp.MemberList")
    (directive,) = Objects.properties(node)["members"].values
    group = Objects.properties(Terms.kind_of(Objects.properties(Terms.kind_of(directive)[1])["directive"])[1])["group"]
    assert group.as_schema().name == "Expressions.C.Preprocessor.Group"  # C's union, C++'s branch
    assert Terms.kind_of(group)[0] == "cpp_membergroup"
# Run again, the upsert finds what it wrote; a bare builder needs the store
STORE.transaction(Parse.parse(source, "shape.cpp"))
with STORE.transaction():
    assert len(Objects.properties(unit(STORE, "shape.cpp"))["body"].values) == 1
with raises(TypeError):
    STORE.Mutations.OfTransaction(Parse.parse("int x;", "bare.cpp"))
built = Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse("int x;", "bare.cpp", STORE)))
assert "bare.cpp" in json.dumps(built)
assert parse("") == [] and kinds(parse("int x;\n\n\nint y;\n")) == ["SimpleDeclaration", "SimpleDeclaration"]
assert [s["line"] for s in parse("int x;\n\n[[deprecated]]\nint y;\n")] == [1, 3]
print("ok")

<a id="XPRS-03"></a>
## XPRS-03 · Literals keep their spelling: integers in each base with C++'s suffixes and separators, floating literals, characters and strings with their prefixes, raw strings with their delimiters, user-defined literals, `true`, `false`, `nullptr`, and adjacent strings

In [ ]:
def literal(text):
    tag, attributes, _ = value(text, shapes=True)
    return tag, attributes


assert literal("1'000ull") == ("literal", {"value": 1000, "text": "1'000ull", "base": 10, "separators": True,
                                           "suffix": "ull"})
assert [literal(t)[1]["base"] for t in ("0x1F", "0b101", "017", "0", "10")] == [16, 2, 8, 8, 10]
assert literal("3uz")[1]["suffix"] == "uz" and literal("1.0f16")[1]["suffix"] == "f16"  # C++23's
assert literal("1.5e-3") == ("literal", {"value": 0.0015, "text": "1.5e-3"}) and literal("0x1.8p3")[1]["value"] == 12
assert literal("'a'") == ("literal", {"value": 97, "text": "'a'", "quote": "'"})
assert literal("u8'x'")[1]["prefix"] == "u8" and literal(r"'\n'")[1]["value"] == 10
assert literal('L"w"')[1] == {"value": "w", "text": 'L"w"', "prefix": "L", "quote": '"'}
assert literal('R"x(a "b")x"')[1] == {"value": 'a "b"', "text": 'R"x(a "b")x"', "quote": '"', "raw": True,
                                       "delimiter": "x"}
assert literal('u8R"(c)"')[1]["prefix"] == "u8" and "delimiter" not in literal('R"(c)"')[1]
assert value("12_km", shapes=True) == ("userdefinedliteral", {"literal": ("literal", {
    "value": 12, "text": "12", "base": 10}), "suffix": "_km"}, [])
assert value('"s"_sv') == '"s"_sv'
assert literal("true") == ("literal", {"value": True, "text": "true"})
assert literal("nullptr") == ("constant", {"singleton": "nullptr", "text": "nullptr"})
assert value('"a" L"b"', shapes=True)[0] == "concatenatedstring"
for text in ("1'000ull", "0x1F", "0b101", "017", "1.5e-3", "'a'", 'R"x(a)x"', "12_km", "true", "nullptr", '"a" "b"'):
    assert value(text) == text, (text, value(text))
assert literal("0x1p99999")[1]["value"] == float("inf")  # beyond the largest
assert value("1f") == "1f"  # `f` after an integer: a user-defined literal's suffix
print("ok")

<a id="XPRS-04"></a>
## XPRS-04 · Expressions: names qualified, templated and operators', every operator with C++'s grouping, casts, `new` and `delete`, lambdas and their captures, folds, `requires`, coroutines' operators, `throw`, `sizeof...`, and GNU's statement expressions

In [ ]:
def tree(text):
    return value(text, shapes=True)


assert tree("std::cout")[1]["name"][0] == "qualifiedname" and tree("::x")[1]["name"][1]["global_scope"] is True
assert tree("f<int>")[1]["name"][0] == "templateid" and first("int operator+(int);")["declarators"][0][
    "declarator"] == "operator+(int)"
assert tree("p->~T()")[2][0][1]["member"][0] == "destructorname"
assert tree("obj.template get<int>()")[2][0][1]["template_keyword"] is True
assert tree("a <=> b")[1] == {"operator": "<=>"} and tree("a.*b")[1] == {"operator": ".*"}
assert tree("static_cast<int>(x)")[:2] == ("namedcastexpression", {"operator": "static_cast", "type": None}) or (
    tree("static_cast<int>(x)")[0] == "namedcastexpression")
assert tree("int(x)")[0] == "functionalcastexpression" and tree("T{x}")[0] == "functionalcastexpression"
assert tree("typeid(x)")[0] == "typeidexpression" and tree("noexcept(f())")[0] == "noexceptexpression"
assert tree("new int[3]()")[0] == "newexpression" and tree("::delete[] p")[1] == {"global_scope": True,
                                                                                  "array": True}
assert tree("throw")[0] == "throwexpression" and tree("sizeof...(Ts)")[0] == "sizeofpackexpression"
assert tree("(... + xs)")[0] == "foldexpression" and tree("(xs + ... + 0)")[1] == {"operator": "+"}
assert tree("f(args...)")[2][1][0] == "packexpansion"
lambda_ = tree("[=, &a, this, *this, n = 1, ...rest = xs]<typename T>(T x) mutable -> T { return x; }")
assert lambda_[0] == "lambdaexpression" and [c[0] for c in lambda_[1]["captures"]] == [
    "defaultcapture", "simplecapture", "thiscapture", "thiscapture", "initcapture", "initcapture"]
assert lambda_[1]["block"]["body"][0]["kind"] == "ReturnStatement"  # its statements, this grammar's
requires = tree("requires(T t) { t.f(); typename T::v; { t.g() } noexcept -> C; requires D<T>; }")
assert [r[0] for r in requires[1]["requirements"]] == ["simplerequirement", "typerequirement",
                                                       "compoundrequirement", "nestedrequirement"]
assert statement("co_yield 1;")["expression"] == "co_yield 1" and statement("co_await x;")["expression"] == (
    "co_await x")
assert tree("({ int t = 1; t; })")[1]["block"]["body"][1]["expression"] == "t"
assert tree("a and not b or c")[1] == {"operator": "||"}  # alternative tokens, normalized (18)
for text in ("a + b * c", "(a + b) * c", "std::vector<int>{1, 2}", "x = y ? 1 : 2", "*p++", "obj->f().g[1]",
             "new (buffer) T(1, 2)", "a[1, 2]", "Point{.x = 1, .y = 2}"):
    assert value(text) == text, (text, value(text))
assert [first(f"int y = {t};")["declarators"][0]["initializer"] for t in ("auto(x)", "auto{x}")] == [
    " = auto(x)", " = auto{x}"]  # C++23's decay-copy
assert value("dynamic_cast<B *>(p)") == "dynamic_cast<B*>(p)"  # an abstract declarator, unspaced
assert value("[&](auto &&x) { return x; }") == "[&](auto &&x) {\n    return x;\n}"  # its block, laid out
assert value("new int") == "new int" and value("sizeof x") == "sizeof x" and first("auto l = [x, y...] {};")["declarators"][0]["initializer"] == (
    " = [x, y...] {}")
assert value("a.~decltype(x)()") == "a.~decltype(x)()" and value("&S::operator\"\"_km") == "&S::operator\"\"_km"
assert value("S::template X<int>::y") == "S::template X<int>::y" and value("'c'_x") == "'c'_x"
assert value("[]() requires C<T> {}") == "[]() requires C<T> {}" and value("[]() [[nodiscard]] {}") == (
    "[]() [[nodiscard]] {}")
print("ok")

<a id="XPRS-05"></a>
## XPRS-05 · Declarations: specifiers and attributes in source order, declarators (references, pointers to members, functions' qualifiers, exception specifications and trailing return types), C++'s initializers, templates and their parameters, concepts, aliases, using-declarations and directives, namespaces, linkage specifications and explicit instantiations

In [ ]:
declaration = first("[[nodiscard]] static constexpr const unsigned long x = 0, &y = x, z{1}, w(2);")
assert declaration["attributes"] == ["[[nodiscard]]"]
assert declaration["specifiers"] == ["static", "constexpr", "const", "unsigned", "long"]
assert [d["declarator"] for d in declaration["declarators"]] == ["x", "&y", "z", "w"]
assert [d["initializer"] for d in declaration["declarators"]] == [" = 0", " = x", "{1}", "(2)"]
assert first("int C::*p;")["declarators"][0]["declarator"] == "C::*p"
assert first("auto f(int) const & noexcept -> int;")["declarators"][0]["declarator"] == (
    "f(int) const & noexcept -> int")
assert first("template <class T> void f(T) requires C<T>;", shapes=True)["declaration"]["declarators"][0][
    "requires"][0] == "idexpression"  # a trailing requires-clause, the declarator's (23)
template = first("template <typename T, int N = 3, template <class> class W, typename... R> requires C<T> struct X;")
assert template["kind"] == "TemplateDeclaration" and template["parameters"] == [
    "typename T", "int N = 3", "template <class> class W", "typename... R"] and template["requires"] == "C<T>"
assert "parameters" not in first("template <> struct T<int> {};")  # an explicit specialization
assert first("template <class T> concept C = true;")["declaration"]["kind"] == "ConceptDefinition"
assert first("using V = std::vector<int>;") == {"kind": "AliasDeclaration", "line": 1, "name": "V",
                                                 "type": "std::vector<int>"}
assert kinds(parse("using namespace std;\nusing std::cout;\nusing enum E;\nnamespace a = b::c;")) == [
    "UsingDirective", "UsingDeclaration", "UsingEnumDeclaration", "NamespaceAliasDefinition"]
namespace = first("inline namespace a::inline b { int x; }")
assert namespace["inline"] is True and [n.get("inline") for n in namespace["names"]] == [None, True] and kinds(
    namespace["body"]) == ["SimpleDeclaration"]
assert first('extern "C" { int f(); }')["braced"] is True and kinds(first('extern "C" int f();')["body"]) == [
    "SimpleDeclaration"]
assert same("template class V<int>;\nextern template class V<float>;\n")
assert same("template <typename T>\nconcept Number = std::is_arithmetic_v<T>;\n")
assert same("int (*pick(int which))(int);\nint &&forwarded = 1;\nchar *const *volatile p;\n")
assert same("[[deprecated(\"use g\")]] int f(int a, int b = 2, ...) __attribute__((noinline));\n")
for text in ("struct D : [[a]] B {};\n", "enum {\n    A\n};\n", "typename T::template X<int> y;\n",
             "decltype(x)::type y;\n", "namespace n = ::m;\nnamespace o = m;\n", "void f() throw(int);\n",
             "template <class T>\nvoid f(T t = T{}) {}\n", "template <Number T>\nvoid f(T);\n",
             "template <int N>\nstruct X;\n", "template <class T>\nstruct S<T>::X;\n", "int x [[a]];\n",
             "S::operator int() const {\n    return 0;\n}\n",
             "struct S {\n    operator int&();\n    operator int&&();\n    int a = 1, b = 2;\n    class N {};\n};\n"):
    assert same(text), (text, printed(text))
assert printed("struct D : /* dropped */ B {};") == "struct D : B {};\n"
for text in ("[[deprecated]];\n", "template void f<int>(int);\n", "S::S() try {} catch (...) {}\n", "template <C auto N>\nstruct X;\n", "template <class T>\nint x = 1;\n",
             "struct D : B<int>... {};\n", "struct S {\n    template <class T>\n    friend class F;\n};\n",
             "struct S {\n    int a, b : 2;\n    enum E {\n        A\n    };\n    union {\n        int i;\n    };\n};\n"):
    assert same(text), (text, printed(text))
print("ok")

<a id="XPRS-06"></a>
## XPRS-06 · Statements: conditions that declare, initializers of `if`, `switch` and range-based `for`, `if constexpr`, `try` and its handlers, `co_return`, `throw`, labels and their substatements

In [ ]:
conditional = statement("if constexpr (auto it = m.find(1); it != m.end()) use(it);")
assert conditional["constexpr"] is True and conditional["initializer"]["kind"] == "SimpleDeclaration"
assert conditional["condition"] == "it != m.end()"
assert statement("if (int x = f(1)) {}")["condition"]["declarators"][0]["initializer"] == " = f(1)"  # (26)
switched = statement("switch (int m = get(); m) { case 1: break; default: ; }")
assert switched["initializer"]["kind"] == "SimpleDeclaration" and kinds(switched["body"]["body"]) == [
    "CaseStatement", "DefaultStatement"]
assert statement("while (int y = g()) {}")["condition"]["kind"] == "SimpleDeclaration"
assert [statement(t)["initializer"] for t in ("if (g(); a) {}", "switch (g(); a) {}", "for (i = 0;;) {}")] == [
    "g()", "g()", "i = 0"]
ranged = statement("for (std::vector<int> v = {1}; int x : v) use(x);")
assert ranged["kind"] == "RangeForStatement" and ranged["declaration"]["declarators"][0]["declarator"] == "x"
assert ranged["range"] == "v" and ranged["initializer"]["kind"] == "SimpleDeclaration"
tried = statement("try { f(); } catch (const E &e) { throw; } catch (...) {}")
assert tried["kind"] == "TryStatement" and [h["parameter"] for h in tried["handlers"]] == ["const E &e", "..."]
assert kinds(tried["handlers"]) == ["Handler", "Handler"] and tried["handlers"][0]["body"]["body"][0][
    "expression"] == "throw"
assert statement("co_return 1;") == {"kind": "CoReturnStatement", "line": 2, "value": "1"}
assert same("void f() {\n    if (a) {\n        b();\n    } else if (c)\n        d();\n    else {}\n"
            "    for (auto &x : xs) {}\n    for (int i = 0; i < n; ++i)\n        continue;\n    do {\n"
            "        a++;\n    } while (a < 10);\nl:\n    goto l;\n}\n")
print("ok")

<a id="XPRS-07"></a>
## XPRS-07 · Classes: members in their `members`, access specifiers, constructors' member initializers, function-try-blocks, `= default`, `= delete` and `= 0`, virt-specifiers, friends, nested types, bit-fields, member templates and `using`s; bases; enumerations, scoped and with a fixed type

In [ ]:
held = members("""class [[nodiscard]] V final : public B<T>, virtual private O, protected M<N>... {
public:
    using value_type = T;
    V() : data{}, count(0) {}
    V(const V &) = delete;
    virtual ~V() = default;
    virtual void draw() const = 0;
    void update() override final;
    friend class F;
    friend V operator+(const V &, const V &);
    template <typename U> U convert() const;
    struct Node { int x; };
    int bits : 4;
    static_assert(true);
protected:
    using B<T>::f;
private:
    int count;
};""")
assert [m[0] if isinstance(m, tuple) else m["kind"] for m in held] == [
    "AccessSpecifier", "AliasDeclaration", "FunctionDefinition", "FunctionDefinition", "FunctionDefinition",
    "MemberDeclaration", "MemberDeclaration", "MemberDeclaration", "MemberDeclaration", "TemplateDeclaration",
    "MemberDeclaration", "MemberDeclaration", "StaticAssertDeclaration", "AccessSpecifier", "UsingDeclaration",
    "AccessSpecifier", "MemberDeclaration"]
assert held[0] == {"kind": "AccessSpecifier", "line": 2, "access": "public"}
assert [i["member"] for i in held[2]["initializers"]] == ["data", "count"]
assert held[3]["body"]["kind"] == "DeletedBody" and held[4]["body"]["kind"] == "DefaultedBody"
assert held[5]["declarators"][0]["pure"] is True and held[6]["declarators"][0]["virt_specifiers"] == [
    "override", "final"]
assert held[7]["specifiers"] == ["friend", "class F"] and held[11]["declarators"][0]["bitfield"] == "4"
classed = first("class V final : public B<T>, virtual private O {};")["specifiers"][0]
assert classed == "class V final : public B<T>, virtual private O {}"
assert same("struct S {\n    S() try : x(1) {} catch (...) {}\n    int y{2};\n};\n")
assert same("enum class Mode : unsigned char {\n    Off,\n    // the default\n    On\n};\nenum struct Opaque : int;\n")
print("ok")

<a id="XPRS-08"></a>
## XPRS-08 · Modules, directives and comments: module and import declarations where they stood, `export`ed declarations and blocks; directives as statements and members, a conditional's lines its `group`; comments on their lines or trailing

In [ ]:
modules = parse('module;\n#include <a>\nexport module m.n:p;\nimport std;\nexport import :q;\nimport <v>;\n'
                'import "h";\nexport int f();\nexport {\n    int g();\n}\nmodule :private;\n')
assert kinds(modules) == ["GlobalModuleFragment", "Directive", "ModuleDeclaration", "ImportDeclaration",
                          "ImportDeclaration", "ImportDeclaration", "ImportDeclaration", "ExportDeclaration",
                          "ExportDeclaration", "PrivateModuleFragment"]
assert modules[2] == {"kind": "ModuleDeclaration", "line": 3, "export": True, "name": "m.n", "partition": "p"}
assert modules[5]["header"] == "v" and modules[5]["system"] is True and modules[6]["header"] == "h"
assert modules[8]["braced"] is True and kinds(modules[7]["body"]) == ["SimpleDeclaration"]
conditional = first("#if A\nint a;\n#else\nint b;\n#endif\n", shapes=True)["directive"]
assert conditional[0] == "ifdirective" and kinds(conditional[1]["group"]["body"]) == ["SimpleDeclaration"]
assert same("#ifdef A\n#elifdef B\nint b;\n#elifndef C\n#endif\n")
assert same("// line\n/* block */\nint x; // trailing\nnamespace n {\n    // inside\n}\n")
assert same("struct S {\n#ifdef A\n    int a;\n#else\n    int b;\n#endif\n};\n")
print("ok")

<a id="XPRS-09"></a>
## XPRS-09 · Invalid syntax is kept, never raised: an `Error` where a statement, a declaration or a member stands, an `error` or `missing` term within an expression, each as written; only text that is no text raises

In [ ]:
mixed = parse("int a = 1;\nint b = ;\nint c;\n")
assert kinds(mixed) == ["SimpleDeclaration", "Error", "SimpleDeclaration"]
assert mixed[1] == {"kind": "Error", "line": 2, "text": "int b = ;"}
assert first("int b = (1 + );", shapes=True)["declarators"][0]["initializer"] == " = (1 + )"  # a `missing` term
inside = body("a = 1;\nif (a {}\nb = 3;")
assert inside[1] == {"kind": "Error", "line": 3, "text": "if (a {}\nb = 3;", "body": [  # as tree-sitter recovers
    {"kind": "ExpressionStatement", "line": 4, "expression": "b = 3"}]}
assert value("x = = 2", shapes=True)[:2] == ("error", {"text": "x = = 2"})
assert printed("struct s { int a; + };") == "struct s {\n    int a;\n    +\n};\n"
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
    Parse.parse("int a;\nx = 1;\n", "invalid.cpp", STORE))))
fresh = Cpp.store()
fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
with fresh.transaction():
    nodes = [Terms.kind_of(n)[1] for n in Objects.properties(unit(fresh, "invalid.cpp"))["body"].values]
    assert [node.as_schema().name for node in nodes] == ["Programs.Cpp.SimpleDeclaration", "Programs.Cpp.Error"]
with raises(ParseError, match="not text: a lone surrogate"):
    Parse.parse("int x;\n  \udfff", "failing.cpp")
try:
    Parse.parse("int x;\n  \udfff", "failing.cpp")
except ParseError as raised:
    assert (raised.line, raised.column) == (2, 3)
for text in ("import a.;", "import ;", "import std", "int x;\nexport", "struct S { int x : 4 = 1",
             "enum E { A [[a", '_Pragma("once");', "auto t = typeid(int +);", "auto t = typeid(a, b);",
             "auto c = static_cast<int>(a, b);", "void f() { try {} catch (int a, int b) {} }",
             "void f(int, );", 'int f() asm("a") asm("b");', "int x = default;", "template <class T> namespace n {}",
             "using a, +;", "struct S { int x : 4 = + ; };", "struct S { virtual void f() = 0 {} };",
             "struct S { namespace n {} };", "struct D : B, {};", "int * [[a", '_Pragma("once");\nint b = ;',
             'void f() asm("x") {}', "template <class T> using namespace std;",
             "struct S { using namespace std; };", "int x, ;"):
    assert error(text), text
print("ok")

<a id="XPRS-10"></a>
## XPRS-10 · Where tree-sitter-cpp departs from C++: each place `Parse`'s docstring enumerates, by its number, with the tree read and the text printed back

In [ ]:
# What tree-sitter-cpp cannot parse, or misreads wherever it stands, which the pre-pass rewrites
# 1. Module declarations
assert kinds(parse("module;\nimport std;\nmodule :private;")) == [
    "GlobalModuleFragment", "ImportDeclaration", "PrivateModuleFragment"]
assert kinds(parse("export module;")) == ["Error"] and same("export module m;\nexport import :p;\n")
# 2. `export`
exported = first("export template <class T> void h(T);")
assert exported["kind"] == "ExportDeclaration" and kinds(exported["body"]) == ["TemplateDeclaration"]
# 3. `extern template`
assert first("extern template class V<float>;")["extern"] is True
# 4. `if consteval`
consteval = statement("if !consteval { a(); }")
assert consteval["consteval"] is True and consteval["negated"] is True and "condition" not in consteval
assert same("void f() {\n    if consteval {\n        a();\n    } else {}\n}\n")
# What tree-sitter-cpp cannot parse, which the pre-pass rewrites where it found an error
# 5. An explicit object parameter
assert first("struct S { void f(this S &self); };")["specifiers"][0].count("this S &self") == 1
assert members("struct S { void f(this S &self); };")[0]["declarators"][0]["declarator"] == "f(this S &self)"
# 6. A using-declaration of several declarators
assert first("using std::a, typename T::b, std::c;")["declarators"] == [
    {"kind": "UsingDeclarator", "name": "std::a"}, {"kind": "UsingDeclarator", "typename_keyword": True,
                                                    "name": "T::b"}, {"kind": "UsingDeclarator", "name": "std::c"}]
assert same("using std::map<int, int>::iterator;\n")  # a comma within angle brackets is no declarator's
# 7. A bit-field's default member initializer
bits = members("struct S { int x : 4 = 1; int y : 2 {3}; };")
assert [d["declarators"][0]["initializer"] for d in bits] == [" = 1", "{3}"]
assert same("struct S {\n    int x : 4 = 1;\n    int y : 2 {3};\n    S() : x(1) {}\n};\n")
# 8. An attribute-using-prefix
assert first("[[using gnu: hot, cold]] void f();")["attributes"] == ["[[using gnu: hot, cold]]"]
# 9. Attributes after an enumerator's name, or after a pointer's `*`
assert same("enum E {\n    A [[deprecated]] = 1,\n    B\n};\n")
assert first("int * [[a]] const p = 0;")["declarators"][0]["declarator"] == "*[[a]] const p"
# 10. A label ending a block
assert body("l: }\n{") == [{"kind": "LabeledStatement", "line": 2, "label": "l"}] or kinds(body("l: ")) == [
    "LabeledStatement"]
assert same("void f() {\nl:\n}\n") and error("void f() { l:}")
# 11. `_Pragma` where an item stands
assert kinds(parse('_Pragma("once") int x;')) == ["Directive", "SimpleDeclaration"]
assert same('void f() {\n    _Pragma("omp parallel")\n    g();\n}\n')
# 12. The null directive
assert parse("#\nint x;")[0] == {"kind": "Directive", "line": 1, "directive": "#"}
# 13. Digraphs, and `->*`
assert printed("int a<:3:> = <% 1 %>;\n%:define X 1\n") == "int a[3] = {1};\n#define X 1\n"
assert tree("p ->* m")[1] == {"operator": "->*"} and value("(p->*m)(1)") == "(p->*m)(1)"
# 14. An alias declaration as an init-statement
assert statement("for (using T = int; ;) {}")["initializer"] == {"kind": "AliasDeclaration", "name": "T", "type": "int"}
assert same("void f() {\n    if (using T = int; true) {}\n    for (using U = long; auto v : w) {}\n}\n")  # parsed
assert same("void f() {\n    for (using T = int;;) {}\n    for (using U = long; auto v : w) {}\n}\n")  # pre-passed
assert error("void f() { for (using T = ; ;) {} }") and error("void f() { for (using std::x; ;) {} }")
# 15. A static lambda
assert value("[](int x) static constexpr { return x; }", shapes=True)[1]["declarator"][1]["specifiers"] == [
    ("declspecifier", {"keyword": "static"}), ("declspecifier", {"keyword": "constexpr"})]
assert same("auto l = []<class T>(T x) static -> int {};\n") and printed("auto l = [] static {};") == (
    "auto l = []() static {};\n")
assert [a[1]["attributes"][0][1]["name"] for a in value("[] [[nodiscard]] [[deprecated]] { return 1; }", shapes=True)[
    1]["attributes"]] == ["nodiscard", "deprecated"]
assert same("auto m = [] [[nodiscard]] {\n    return 1;\n};\n") and error("int b = a[1] [[x]];")
assert error("auto m = [] [[a") and error("(a) static int y;")  # unclosed; a `)` that is no lambda's
# 16. A conversion function's name within an expression
assert value("x.operator int()") == "x.operator int()" and value("p->operator std::vector<int> *()") == (
    "p->operator std::vector<int>*()")
assert statement("auto b = operator bool();")["declarators"][0]["initializer"] == " = operator bool()"
assert same("void f() {\n    auto b = operator bool();\n    auto c = &S::operator int;\n}\n")
assert value("x.operator\n    int()") == "x.operator int()" and error("auto y = x.operator int")
# 17. GNU's extensions, as C's pre-pass rewrites them
assert statement("switch (x) { case 1 ... 3: break; }")["body"]["body"][0]["last"] == "3"
assert statement("goto *p;") == {"kind": "GotoStatement", "line": 2, "target": "p"}
assert first('register int r asm("r1") = 3;')["declarators"][0]["declarator"] == 'r asm("r1")'
assert first("int x __attribute__((aligned(16))) = 0;")["declarators"][0]["declarator"] == (
    "x __attribute__((aligned(16)))")
assert same('void f() {\n    switch (x) {\n        case 1 ... 3:\n            goto *p;\n    }\n}\n')
# What tree-sitter-cpp misreads, which the converter corrects
# 18. An explicit instantiation definition
assert first("template class V<int>;") == {"kind": "ExplicitInstantiation", "line": 1, "declaration": {
    "kind": "SimpleDeclaration", "specifiers": ["class V<int>"]}}
# 19. `typeid` of a type-id, and the operators read as calls
assert first("auto t = typeid(int);", shapes=True)["declarators"][0]["initializer"] == " = typeid(int)"
path = next(PATHS)
STORE.transaction(Parse.parse("auto t = typeid(int);", path))
with STORE.transaction():
    (declaration,) = Objects.properties(unit(STORE, path))["body"].values
    (declarator,) = Objects.properties(Terms.kind_of(declaration)[1])["declarators"].values
    equal = Terms.kind_of(Objects.properties(declarator)["initializer"])[1]
    assert shape(Objects.properties(equal)["value"]) == ("typeidexpression", {}, [("typeid", {
        "specifiers": [("primitivetypespecifier", {"keyword": "int"})]}, [])])
assert tree('_Pragma("x")')[0] == "pragmaexpression"
# 20. A function deleted or defaulted outside a class
assert first("int f() = delete;")["body"]["kind"] == "DeletedBody"
assert same("int f() = delete;\nS::S() = default;\n")
# 21. An unnamed bit-field
assert members("struct S { int : 3; };")[0]["declarators"] == [{"kind": "MemberDeclarator", "bitfield": "3"}]
# 22. Alternative tokens
assert value("a bitand b") == "a & b" and value("x not_eq y") == "x != y"
# 23. Casts read as calls
assert [tree(t)[0] for t in ("int(x)", "static_cast<int>(x)", "T{x}")] == [
    "functionalcastexpression", "namedcastexpression", "functionalcastexpression"]
# 24. `.*` and `->*`
assert tree("a.*m")[0] == "binaryexpression"
# 25. `NULL`
assert tree("NULL") == ("idexpression", {"name": ("identifier", {"spelling": "NULL"})}, [])
# 26. A conversion function's declarator
assert members("struct S { explicit operator T *() const; };")[0]["declarators"][0]["declarator"] == (
    "operator T*() const")
# 27. A declarator's trailing requires-clause and virt-specifiers
assert members("struct S { void f() override; };")[0]["declarators"][0]["virt_specifiers"] == ["override"]
assert first("template <class T> void f() requires C<T> {}")["declaration"]["requires"] == "C<T>"
# 28. The pure-specifier
assert members("struct S { virtual void f() = 0; };")[0]["declarators"][0]["pure"] is True
# 29. An attribute declaration, and GNU's attributes on a null statement
assert statement("[[fallthrough]];")["kind"] == "AttributeDeclaration"
assert statement("__attribute__((fallthrough));")["kind"] == "AttributedStatement"
# 30. A condition's declaration
assert statement("while (int y = g()) {}")["condition"]["declarators"][0]["initializer"] == " = g()"
# 31. GNU's asm label, and `__restrict`
assert first('int f(void) asm("g");')["declarators"][0]["declarator"] == 'f(void) asm("g")'
assert first("int * __restrict r;")["declarators"][0]["declarator"] == "*__restrict r"
# 32. A pointer to member's declarator
assert first("int C::*p;", shapes=True)["declarators"][0]["declarator"] == "C::*p"
# 33. A constraint
assert first("template <class T> requires (A<T> && B<T>) || C<T> void f();")["requires"] == (
    "(A<T> && B<T>) || C<T>")
# 34. `defined X`, and labels as values
assert printed("#if defined X\n#endif\n") == "#if defined(X)\n#endif\n"
assert first("void *p = &&l;", shapes=True)["declarators"][0]["initializer"] == " = &&l"
# 35. A type's keyword as a macro's argument
assert not error("void f() { int x = va_arg(ap, int); }") and same("void f() {\n    int x = va_arg(ap, int);\n}\n")
# 36. `auto(x)` and `auto{x}`
assert [tree(f"y = {t}")[2][1][1]["type"] for t in ("auto(x)", "auto{x}")] == [("placeholdertypespecifier", {})] * 2
assert statement("auto c = auto(x) + auto{x};")["declarators"][0]["initializer"] == " = auto(x) + auto{x}"
# What tree-sitter-cpp reads but C++ rejects, which the converter keeps invalid syntax
# 37. Statements at namespace scope
assert kinds(parse("x = 1;\n{}\n;")) == ["Error", "Error", "EmptyDeclaration"]
# 38. C's constructs
for text in ("int x = _Generic(a, int: 1);", "auto p = (struct P){1, 2};", "int a[] = {[0] = 1};",
             "void f(int a[static 3]);", "int * restrict p;", "_Noreturn void f();"):
    assert error(text), text
assert kinds(parse("int old(a) int a; { return a; }"))[0] == "Error"
# 39. Literals C++ rejects
assert error("int a = 08;") and error("char c = '\\x';")
assert first("auto d = 1uu;")["declarators"][0]["initializer"] == " = 1uu"  # a user-defined literal's suffix, `uu`
# 40. A closing directive where none is open
assert kinds(parse("#endif")) == ["Error"]
# 41. Virt-specifiers where no member is declared, and a definition declaring no function
assert kinds(parse("int f() override;")) == ["Error"] and kinds(parse("int m { return 0; }")) == ["Error"]
# 42. Microsoft's extensions
assert error("int __cdecl f();") and error("void (__cdecl *f)();")
# 43. `_Pragma` of no one string literal
assert error("void f() { _Pragma(x); }")
# What mbse-expressions' C++ dialect cannot hold, which the converter keeps invalid syntax
# 44. Structured bindings
assert kinds(parse("auto [a, b] = p;")) == ["Error"]
# 45. A conditional `explicit`
assert members("struct S { explicit(true) S(int); };")[0]["kind"] == "Error"
assert members("struct S { explicit S(int); };")[0]["specifiers"] == ["explicit"]
# 46. `__has_include`
assert kinds(parse("#if __has_include(<x>)\n#endif")) != ["Directive"]
# 47. A user-defined literal of adjacent strings
assert error('auto z = "a" "b"_s;')
# What stays as tree-sitter-cpp reads it
# 48. Typedef names, template names and concepts: as tree-sitter decides
assert first("template <Number T> void f(T);")["parameters"] == ["Number T"]  # a constant's parameter
assert statement("T * x;")["kind"] == "SimpleDeclaration" and statement("a < b > (c);")["kind"] in (
    "ExpressionStatement", "SimpleDeclaration")
# 49. A conditional directive that splits a construct
assert error("void f() {\n#ifdef A\n}\n#endif\n")
print("ok")